# Animaciones y trayectorias energéticas

Visualiza la marcha esquemática de kinesina, crea animaciones y superpone trayectorias cinéticas sobre un mapa de energía ilustrativo.

> **Nota científica:** KineSIM es un prototipo de modelado cinético fenomenológico. Los parámetros iniciales son ilustrativos y los mapas de energía no están calibrados experimentalmente para kinesina.

## Resultados de aprendizaje

Este laboratorio transforma una trayectoria cinética en dos visualizaciones: una animación esquemática de dos cabezas y una trayectoria proyectada sobre un paisaje energético. Aprenderás qué información procede del simulador, qué elementos son reconstrucciones gráficas y qué significa asignar una coordenada conformacional a cada estado.

**Advertencia central:** la simulación cinética actual registra una sola coordenada axial del motor y eventos discretos. No integra por separado las posiciones de las dos cabezas, los neck linkers ni la carga.

## Instalación desde PyPI

Este notebook utiliza la distribución publicada de **KineSIM**. No modifica `sys.path` ni importa código desde la carpeta `src/`.

- Nombre de distribución: `kinesim` (PyPI no distingue mayúsculas).
- Namespace de importación: `protein_motor_sim` (se conserva por compatibilidad).
- Versión fijada para reproducibilidad: `0.1.2`.

Ejecuta la siguiente celda con acceso a Internet. Si la versión todavía no aparece en PyPI, primero debe publicarse con Twine. Para probar una subida en TestPyPI, consulta las instrucciones de instalación de este README; no mezcles los índices de TestPyPI y PyPI en una misma resolución de dependencias.


In [ ]:
%pip install --quiet "kinesim[notebooks]==0.1.2"


## 1. Objetivo

Este notebook crea animaciones esquemáticas de la marcha de kinesina y visualiza una trayectoria cinética sobre un paisaje energético. Las posiciones individuales de las cabezas en la animación se reconstruyen gráficamente: la simulación cinética solo produce la coordenada del motor y sus eventos.

## De una trayectoria a una animación

La trayectoria contiene tiempos, posiciones del centro del motor, nombres de estados y eventos. La rutina de animación utiliza esa información para reconstruir una escena didáctica con dos cabezas, cuello, tallo, carga y pista.

Durante un evento `forward_step` o `backward_step`, el dibujo interpola una de las cabezas mediante un arco. La cabeza móvil se infiere a partir del orden relativo de las posiciones representadas. Los demás eventos actualizan las etiquetas bioquímicas, pero no generan por sí solos una deformación molecular calculada.

In [ ]:
from importlib.metadata import version as distribution_version
print("KineSIM distribution:", distribution_version("kinesim"))

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display

from protein_motor_sim import Environment, GillespieSimulator, Kinesin1
from protein_motor_sim.models.kinesin.parameters import KinesinParameters
from protein_motor_sim.visualization.animation import (
    AnimationStyle, animate_kinesin_walk, save_animation
)
from protein_motor_sim.mechanics.energy_landscape import EnergyLandscape
from protein_motor_sim.visualization.energy_maps import (
    plot_trajectory_on_energy_landscape, animate_trajectory_on_energy_landscape
)

## 2. Generar una trayectoria suficientemente larga

Reducimos la tasa basal de desprendimiento para facilitar una animación con varios eventos. Este cambio es solo una elección de demostración; no es un ajuste a datos experimentales.

In [ ]:
parameters = KinesinParameters(detachment_rate_s=0.03)
motor = Kinesin1(parameters=parameters)
environment = Environment(atp_concentration=1e-4, external_force_N=0.0, temperature_K=310.0)
trajectory = GillespieSimulator().run(
    motor, environment, t_max=0.2, seed=415, max_events=100000
)
print("Registros:", len(trajectory.times_s))
print("Eventos:", trajectory.events.count("forward_step"), "avances;",
      trajectory.events.count("backward_step"), "retrocesos")

### Por qué modificamos la tasa de desprendimiento

Se reduce `detachment_rate_s` únicamente para conseguir una trayectoria suficientemente larga para la animación. Esto es un ajuste de demostración, no una estimación experimental ni una afirmación sobre la processividad real de kinesina.

La concentración de ATP y la semilla también se fijan para que el ejemplo sea reproducible. Cambiar la semilla produce otra realización posible.

## 3. Animación de dos cabezas

La escena dibuja subunidades de tubulina, cabezas A y B, cuello conector, tallo, carga, estela y anotaciones de tiempo/evento. El eje vertical es esquemático y no está a escala.

In [ ]:
style = AnimationStyle(
    step_size_nm=motor.step_size_nm,
    frames_per_step=4,
    interval_ms=60,
    trail_length=15,
    show_trail=True,
    show_time=True,
)
walk_animation = animate_kinesin_walk(trajectory, style=style, repeat=False)
display(HTML(walk_animation.to_jshtml()))

### Qué significan los parámetros visuales

- `step_size_nm`: separación nominal asociada a un paso del motor.
- `frames_per_step`: número de fotogramas usados para interpolar el salto; más fotogramas producen una animación visualmente más suave, no una mayor resolución física.
- `interval_ms`: tiempo de presentación de cada fotograma.
- `trail_length`: cantidad de posiciones recientes mostradas en la estela.
- `show_trail` y `show_time`: activan o desactivan elementos informativos.

El eje vertical y la forma del arco son diagramáticos y no están a escala estructural. La escena no constituye una simulación atomística, una dinámica de Langevin ni una solución mecánica de las dos cabezas.

## 4. Superponer la trayectoria sobre el paisaje

La coordenada `q` no está en la trayectoria cinética actual. Para mostrarla en el mapa tenemos que asignar explícitamente una coordenada ilustrativa a cada estado. Esto no es una medición de conformación molecular.

In [ ]:
landscape = EnergyLandscape(
    period_nm=8.2,
    spatial_barrier_kBT=2.5,
    conformational_barrier_kBT=3.0,
    coupling_kBT=0.6,
    opposing_force_pN=0.0,
    temperature_K=310.0,
)
state_to_q = {
    "bound_apo": 0.15,
    "atp_bound": 0.45,
    "post_hydrolysis": 0.80,
    "detached": 0.50,
}
fig, ax = plt.subplots(figsize=(9, 5))
plot_trajectory_on_energy_landscape(
    trajectory, landscape, state_coordinates=state_to_q, ax=ax
)
plt.show()

## Proyección de estados sobre la coordenada conformacional

La trayectoria cinética no contiene una variable $q$. Por eso el notebook define manualmente el diccionario `state_to_q`, que asigna un valor entre 0 y 1 a cada nombre de estado.

Esta asignación es **ilustrativa**: permite colocar los registros $(x(t),q(t))$ sobre una superficie para explorar cómo se lee un mapa 2D, pero no es una medición de la conformación ni una variable calculada por Gillespie.

## 5. Animar la trayectoria sobre el mapa

El marcador representa el estado y la posición registrados por el modelo cinético. El mapa energético es una superficie fenomenológica independiente; superponerlos es una visualización exploratoria, no una dinámica derivada del gradiente de energía.

In [ ]:
landscape_animation = animate_trajectory_on_energy_landscape(
    trajectory,
    landscape,
    state_coordinates=state_to_q,
    interval_ms=100,
)
display(HTML(landscape_animation.to_jshtml()))

### Interpretar la animación sobre el paisaje

El marcador muestra la posición axial y la coordenada $q$ asignada al estado registrado en cada evento. El fondo es la función $U(x,q)$ configurada en el notebook.

La superposición no significa que el motor se haya movido siguiendo $-\nabla U$ ni que las tasas cinéticas se hayan calculado a partir de las barreras dibujadas. Son dos representaciones independientes colocadas en una misma figura con finalidad exploratoria.

## 6. Exportar un GIF (opcional)

Para exportar GIF instala Pillow (`python -m pip install pillow`). Para MP4 se necesita FFmpeg. Los archivos se guardarán en `results/` dentro del proyecto.

In [ ]:
from pathlib import Path
OUTPUT_DIR = Path.cwd() / "kinesim_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

gif_path = OUTPUT_DIR / "kinesin_walk.gif"
try:
    saved_path = save_animation(walk_animation, gif_path, fps=15, dpi=90)
    print("GIF guardado en:", saved_path)
except (ImportError, RuntimeError, ValueError) as error:
    print("No se pudo exportar el GIF:", error)
    print("Instala Pillow para GIF o FFmpeg para MP4.")

## Exportación y reproducibilidad

La animación puede mostrarse directamente en Jupyter mediante HTML. `save_animation` permite guardarla como GIF o MP4 según las herramientas disponibles: Pillow para GIF y FFmpeg para MP4. Los fotogramas son interpolaciones de la escena, no pasos temporales microscópicos adicionales.

Para comparar animaciones conserva la semilla, las condiciones del entorno, los parámetros del motor y el estilo visual.

## Referencias

- Carter, N. J., & Cross, R. A. (2005). *Mechanics of the kinesin step*. Nature, 435, 308–312. [https://doi.org/10.1038/nature03528](https://doi.org/10.1038/nature03528)
- Block, S. M. (2007). *Kinesin motor mechanics: Binding, stepping, tracking, gating, and limping*. Biophysical Journal, 92(9), 2986–2995. [https://doi.org/10.1529/biophysj.106.100677](https://doi.org/10.1529/biophysj.106.100677)

## 7. Ideas para experimentar

- Cambia `frames_per_step` para controlar la interpolación visual.
- Cambia la semilla para obtener otra trayectoria.
- Aumenta la carga y compara las secuencias de eventos.
- Cambia el mapeo `state_to_q` y observa cómo cambia la proyección sobre el paisaje.

Recuerda: la animación es una reconstrucción didáctica; no se simula la geometría atomística ni la dinámica independiente de ambas cabezas.